# 04 — BoW and TF-IDF baselines

Prepared 9 November 2026 (Asia/Calcutta). **Generated, unexecuted notebook.**

Compare train-only sparse Logistic Regression and Linear SVM before Transformer fine-tuning.

Inputs: configs/training/imdb_binary_tfidf.yaml for the binary benchmark; ml/sentiment_tfidf/config.yaml for genuine three-class annotations.

Use identical frozen role manifests for comparable experiments. Change only the estimator or feature representation per planned ablation; never select hyperparameters using test results.

No datasets, dependencies or model weights are fetched. No model is trained by running all cells with the default flag. Fill only real, approved local paths. Do not store private review text in committed outputs. Source presence is not Phase 2/3 acceptance.

Read [the Phase 3 runbook](../documentation/phase3/RUNBOOK.md) first.

In [ ]:
from pathlib import Path
import sys

# Explicit opt-in for later local inspection; this delivery has not executed cells.
RUN_APPROVED_EXPERIMENTS = False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "ARCHITECTURE.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the D:/BDA_TA workspace.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workspace:", ROOT)
print("Experiment execution enabled:", RUN_APPROVED_EXPERIMENTS)

In [ ]:
CONFIG_PATH = ROOT / "configs/training/imdb_binary_tfidf.yaml"
if RUN_APPROVED_EXPERIMENTS:
    from ml.common.config import load_config
    from ml.sentiment_tfidf.train import preflight
    readiness = preflight(load_config(CONFIG_PATH))
    print(readiness)
else:
    print("Preflight disabled. This notebook never calls fit/train automatically.")

## Controlled experiment matrix

| Experiment | Representation | Estimator | Changed factor |
|---|---|---|---|
| BoW LR | CountVectorizer | Logistic Regression | Counts versus TF-IDF |
| TF-IDF LR | TfidfVectorizer | Logistic Regression | Reference baseline |
| TF-IDF SVM | TfidfVectorizer | LinearSVC | Linear classifier |

Use a fresh config, run directory and explicit run approval per experiment. A bounded n-gram/C grid may be proposed from tuning data, but this notebook does not launch one. Vocabulary and IDF fit on train only; calibration fits on a dedicated calibration role.

## Manual execution later

Copy the reviewed config for Linear SVM, set `tfidf.estimator=linear_svm` and a fresh `output_dir`. Fill real provenance/approval paths and all four manifests. After approval:

```powershell
.\scripts\train_models.ps1 -Component sentiment_tfidf -Action Train -ConfigPath 'configs/training/imdb_binary_tfidf.yaml' -ExecuteApprovedRun
```

The shared implementation saves portable bounded JSON state; no pickle/joblib is loaded. A stopped run is retained and must not be overwritten.

In [ ]:
TRAINING_REPORT_PATHS = []  # Actual training_report.json paths after approved runs.
if RUN_APPROVED_EXPERIMENTS and TRAINING_REPORT_PATHS:
    from ml.common.artifacts import read_json
    reports = [read_json(ROOT / path) for path in TRAINING_REPORT_PATHS]
    reference = reports[0]
    for report in reports:
        assert report["trained"] is True and report["component"] == "tfidf"
        assert report["task"] == reference["task"]
        assert report["manifest_hashes"] == reference["manifest_hashes"]
    print([{"estimator": report["estimator"], "task": report["task"],
            "tuning_metrics": report["tuning_metrics"]} for report in reports])
else:
    print("No actual baseline reports supplied; no comparison or metric is reported.")

## Probability interpretation

SVM margins are not probabilities. The shared implementation fits a positive temperature on the separate calibration split and marks calibration-fit diagnostics. Compare macro-F1, per-class support and confusion matrices alongside calibrated held-out NLL/Brier/ECE. Do not treat calibration-fit metrics as unbiased held-out performance. Zero-vocabulary-feature rows remain visible in reports.

## Interpretation and acceptance

Record measured results, source/model licenses, exact config and manifest hashes, code revision, dependency versions, hardware and unsupported cohorts after the approved runs. Keep the final held-out set separate from model selection. No result has been measured in this notebook. Update the model card from actual reports; missing evidence remains missing.